In [2]:
import torch
from torchvision import datasets, transforms
from torch.utils.data import DataLoader
import torch.nn as nn

In [3]:
DATA_DIR = r"C:\Users\jia65\Downloads\dataset"
IMG_SIZE = 64
BATCH_SIZE = 64

transform = transforms.Compose([
    transforms.Resize(IMG_SIZE),
    transforms.CenterCrop(IMG_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.5, 0.5, 0.5), std=(0.5, 0.5, 0.5))
])

class WikiArtDataset(datasets.ImageFolder):
    def find_classes(self, directory):
        classes, _ = super().find_classes(directory)
        classes = [name for name in classes if not name.startswith(".")]
        class_to_idx = {name: i for i, name in enumerate(classes)}

        return classes, class_to_idx


dataset = WikiArtDataset(
    root=DATA_DIR,
    transform=transform
)


print("Number of Images", len(dataset))
print("Number of classes", len(dataset.classes))

Number of Images 81444
Number of classes 27


In [4]:
# Discriminator
# Input imgae
# Output score

class Discriminator(nn.Module):
    def __init__(self):
        super().__init__()
        self.model = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=4, stride=2, padding=1),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            nn.Conv2d(64, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            nn.Conv2d(128, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            nn.Conv2d(256, 512, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(512),
            nn.LeakyReLU(negative_slope=0.2, inplace=True),
            nn.Conv2d(512, 1, kernel_size=4, stride=1, padding=0),
        )

    def forward(self, x):
        return self.model(x).view(-1, 1)

In [5]:
# Generator

class Generator(nn.Module):
    def __init__(self, latent_dim=100):
        super().__init__()
        self.model = nn.Sequential(
            nn.ConvTranspose2d(latent_dim, 512, kernel_size=4, stride=1, padding=0),
            nn.BatchNorm2d(512),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(512, 256, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(256),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(256, 128, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(128, 64, kernel_size=4, stride=2, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.ConvTranspose2d(64, 3, kernel_size=4, stride=2, padding=1),
            nn.Tanh()
        )

    def forward(self, z):
        return self.model(z)

In [26]:
import os, json, csv, time, random
from pathlib import Path
import numpy as np
import torch
import torch.nn as nn
import torchvision
from torch.utils.data import DataLoader, Subset
from torchvision import transforms
from torchvision.utils import save_image
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display
from tqdm.auto import tqdm
import ignite
from ignite.metrics import FID, InceptionScore

SEED = 42
LATENT_DIM = 100
EPOCHS = 30
BATCH_SIZE = 128
LR = 0.0002
BETAS = (0.5, 0.999)
NUM_WORKERS = 0
EVAL_SAMPLES = 5000
EVAL_BATCH_SIZE = 32
EVAL_EVERY = 5
OUTPUT_DIR = Path(r"C:\Users\jia65\Desktop\ECE285\hw2\gan_results")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
(OUTPUT_DIR / "samples").mkdir(exist_ok=True)
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

split_rng = torch.Generator().manual_seed(SEED)
indices = torch.randperm(len(dataset), generator=split_rng).tolist()
eval_indices = indices[:EVAL_SAMPLES]
train_indices = indices[EVAL_SAMPLES:]
print(f"Training images: {len(train_indices)} | Evaluation images: {len(eval_indices)}")
train_loader = DataLoader(Subset(dataset, train_indices), batch_size=BATCH_SIZE,
                          shuffle=True, num_workers=NUM_WORKERS, drop_last=True,
                          pin_memory=device.type == "cuda")
eval_loader = DataLoader(Subset(dataset, eval_indices), batch_size=EVAL_BATCH_SIZE,
                         shuffle=False, num_workers=NUM_WORKERS)
with (OUTPUT_DIR / "split.json").open("w") as f:
    json.dump({"seed": SEED,
               "train_files": [dataset.samples[i][0] for i in train_indices],
               "eval_files": [dataset.samples[i][0] for i in eval_indices]}, f)

def initialize_weights(m):
    if isinstance(m, (nn.Conv2d, nn.ConvTranspose2d)):
        nn.init.normal_(m.weight, 0.0, 0.02)
        if m.bias is not None:
            nn.init.zeros_(m.bias)
    elif isinstance(m, nn.BatchNorm2d):
        nn.init.normal_(m.weight, 1.0, 0.02)
        nn.init.zeros_(m.bias)

G = Generator(latent_dim=LATENT_DIM).to(device)
D = Discriminator().to(device)
G.apply(initialize_weights)
D.apply(initialize_weights)
criterion = nn.BCEWithLogitsLoss()
optimizer_G = torch.optim.Adam(G.parameters(), lr=LR, betas=BETAS)
optimizer_D = torch.optim.Adam(D.parameters(), lr=LR, betas=BETAS)
fixed_noise = torch.randn(64, LATENT_DIM, 1, 1, generator=torch.Generator().manual_seed(123)).to(device)
config = dict(seed=SEED, epochs=EPOCHS, image_size=64, batch_size=BATCH_SIZE,
              latent_dim=LATENT_DIM, optimizer="Adam", lr=LR, betas=BETAS,
              loss="BCEWithLogitsLoss; D=real+fake", train_images=len(train_indices),
              train_sampling="all images except fixed evaluation split; seed 42",
              train_drop_last=True, eval_images=EVAL_SAMPLES, eval_every=EVAL_EVERY,
              pixel_range=[-1,1], fid="Ignite default Inception output, 1000 dimensions",
              eval_preprocess="[-1,1] to [0,1], PIL uint8 bilinear resize to 299",
              torch=torch.__version__, torchvision=torchvision.__version__, ignite=ignite.__version__,
              generator_parameters=sum(p.numel() for p in G.parameters()),
              discriminator_parameters=sum(p.numel() for p in D.parameters()))
(OUTPUT_DIR / "hyperparameters.json").write_text(json.dumps(config, indent=2))
print(json.dumps(config, indent=2))
real_preview = torch.stack([dataset[i][0] for i in eval_indices[:64]])
save_image((real_preview + 1) / 2, OUTPUT_DIR / "real_images.png", nrow=8)


Device: cuda
GPU: NVIDIA GeForce RTX 3070 Ti
Training images: 76444 | Evaluation images: 5000
{
  "seed": 42,
  "epochs": 30,
  "image_size": 64,
  "batch_size": 128,
  "latent_dim": 100,
  "optimizer": "Adam",
  "lr": 0.0002,
  "betas": [
    0.5,
    0.999
  ],
  "loss": "BCEWithLogitsLoss; D=real+fake",
  "train_images": 76444,
  "train_sampling": "all images except fixed evaluation split; seed 42",
  "train_drop_last": true,
  "eval_images": 5000,
  "eval_every": 5,
  "pixel_range": [
    -1,
    1
  ],
  "fid": "Ignite default Inception output, 1000 dimensions",
  "eval_preprocess": "[-1,1] to [0,1], PIL uint8 bilinear resize to 299",
  "torch": "2.7.1+cu118",
  "torchvision": "0.22.1+cu118",
  "ignite": "0.5.5",
  "generator_parameters": 3577667,
  "discriminator_parameters": 2766529
}


In [27]:
fid_metric = FID(device=device)
is_metric = InceptionScore(device=device)
to_pil = transforms.ToPILImage()
to_tensor = transforms.ToTensor()

def prepare_for_inception(batch):
    batch = ((batch.detach().cpu() + 1) / 2).clamp(0, 1)
    return torch.stack([
        to_tensor(to_pil(img).resize((299, 299), Image.Resampling.BILINEAR))
        for img in batch
    ]).to(device)

@torch.no_grad()
def evaluate_gan():
    was_training = G.training
    G.eval()
    fid_metric.reset()
    is_metric.reset()
    noise_rng = torch.Generator(device=device).manual_seed(2026)
    for real, _ in tqdm(eval_loader, desc="FID / IS", leave=False):
        z = torch.randn(real.size(0), LATENT_DIM, 1, 1,
                        device=device, generator=noise_rng)
        fake_input = prepare_for_inception(G(z))
        real_input = prepare_for_inception(real)
        fid_metric.update((fake_input, real_input))
        is_metric.update(fake_input)
    print("Computing FID covariance distance; this can take a while...", flush=True)
    fid = float(fid_metric.compute())
    score = float(is_metric.compute())
    G.train(was_training)
    return fid, score

@torch.no_grad()
def save_samples(epoch):
    was_training = G.training
    G.eval()
    images = ((G(fixed_noise) + 1) / 2).clamp(0, 1)
    path = OUTPUT_DIR / "samples" / f"epoch_{epoch:03d}.png"
    save_image(images, path, nrow=8)
    G.train(was_training)
    return path

In [28]:
batch_history = []
epoch_history = []
metric_history = []

def write_csv(filename, rows):
    with (OUTPUT_DIR / filename).open("w", newline="") as f:
        writer = csv.DictWriter(f, fieldnames=list(rows[0]))
        writer.writeheader()
        writer.writerows(rows)

def save_plots():
    fig, ax = plt.subplots(figsize=(9, 4))
    ax.plot([r["step"] for r in batch_history], [r["G_loss"] for r in batch_history], label="G", alpha=0.7)
    ax.plot([r["step"] for r in batch_history], [r["D_loss"] for r in batch_history], label="D", alpha=0.7)
    ax.set(xlabel="Training step", ylabel="BCE loss", title="Training loss")
    ax.legend(); fig.tight_layout()
    fig.savefig(OUTPUT_DIR / "loss_curves.png", dpi=180)
    plt.close(fig)
    if metric_history:
        fig, axes = plt.subplots(1, 2, figsize=(10, 4))
        for ax, key, title in zip(axes, ["FID", "IS"], ["Ignite FID-1000 (lower is better)", "Inception Score (higher is better)"]):
            ax.plot([r["epoch"] for r in metric_history], [r[key] for r in metric_history], marker="o")
            ax.set(xlabel="Epoch", ylabel=key, title=title)
        fig.tight_layout()
        fig.savefig(OUTPUT_DIR / "metric_curves.png", dpi=180)
        plt.close(fig)

step_count = 0
for epoch in range(1, EPOCHS + 1):
    start = time.time()
    G.train(); D.train()
    sums = np.zeros(2)
    seen = 0
    bar = tqdm(train_loader, desc=f"Epoch {epoch}/{EPOCHS}")
    for real, _ in bar:
        real = real.to(device, non_blocking=True)
        b = real.size(0)
        D.requires_grad_(True)
        optimizer_D.zero_grad(set_to_none=True)
        real_scores = D(real).view(-1)
        loss_real = criterion(real_scores, torch.ones_like(real_scores))
        loss_real.backward()
        z = torch.randn(b, LATENT_DIM, 1, 1, device=device)
        fake = G(z)
        fake_scores = D(fake.detach()).view(-1)
        loss_fake = criterion(fake_scores, torch.zeros_like(fake_scores))
        loss_fake.backward()
        optimizer_D.step()
        loss_D = loss_real.detach() + loss_fake.detach()
        optimizer_G.zero_grad(set_to_none=True)
        D.requires_grad_(False)
        generated_scores = D(fake).view(-1)
        loss_G = criterion(generated_scores, torch.ones_like(generated_scores))
        loss_G.backward()
        optimizer_G.step()
        D.requires_grad_(True)
        g, d = loss_G.item(), loss_D.item()
        step_count += 1
        batch_history.append(dict(step=step_count, epoch=epoch, G_loss=g, D_loss=d))
        sums += np.array([g, d]) * b
        seen += b
        if step_count % 50 == 0:
            bar.set_postfix(G=f"{g:.3f}", D=f"{d:.3f}")
    row = dict(epoch=epoch, G_loss=float(sums[0]/seen), D_loss=float(sums[1]/seen),
               train_seconds=time.time()-start)
    epoch_history.append(row)
    print(f"Epoch {epoch}: G={row['G_loss']:.4f}, D={row['D_loss']:.4f}, training={row['train_seconds']:.1f}s", flush=True)
    sample_path = save_samples(epoch)
    torch.save(dict(epoch=epoch, G=G.state_dict(), D=D.state_dict(),
                    optimizer_G=optimizer_G.state_dict(), optimizer_D=optimizer_D.state_dict(),
                    fixed_noise=fixed_noise.cpu(), config=config,
                    batch_history=batch_history, epoch_history=epoch_history,
                    metric_history=metric_history, torch_rng=torch.get_rng_state(),
                    cuda_rng=torch.cuda.get_rng_state_all() if torch.cuda.is_available() else None,
                    numpy_rng=np.random.get_state(), python_rng=random.getstate()),
               OUTPUT_DIR / "latest.pt")
    write_csv("batch_losses.csv", batch_history)
    write_csv("epoch_losses.csv", epoch_history)
    save_plots()
    if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
        fid, score = evaluate_gan()
        metric_history.append(dict(epoch=epoch, FID=fid, IS=score, samples=EVAL_SAMPLES))
        write_csv("metrics.csv", metric_history)
        save_plots()
        print(f"Epoch {epoch} | FID (Ignite default, 1000-D): {fid:.4f} | Inception Score: {score:.4f} | N={EVAL_SAMPLES}", flush=True)
        (OUTPUT_DIR / "latest_metrics.json").write_text(json.dumps(metric_history[-1], indent=2))
    display(Image.open(sample_path))

print("Training finished. Results:", OUTPUT_DIR)
display(Image.open(OUTPUT_DIR / "loss_curves.png"))
display(Image.open(OUTPUT_DIR / "metric_curves.png"))
print("Final metrics:", metric_history[-1])


Epoch 1: G=6.6117, D=0.6186, training=1633.5s


Epoch 2: G=4.5196, D=0.5350, training=1384.3s


Epoch 3: G=4.2161, D=0.6149, training=1487.9s


Epoch 4: G=4.0747, D=0.5829, training=1344.7s


Epoch 5: G=3.9123, D=0.5569, training=1331.9s


Computing FID covariance distance; this can take a while...
Epoch 5 | FID (Ignite default, 1000-D): 0.0764 | Inception Score: 2.0000 | N=5000


... (28 outputs left out)

Epoch 30: G=3.5285, D=0.4472, training=1314.1s


Computing FID covariance distance; this can take a while...
Epoch 30 | FID (Ignite default, 1000-D): 0.0280 | Inception Score: 2.6021 | N=5000


Training finished. Results: C:\Users\jia65\Desktop\ECE285\hw2\gan_results


Final metrics: {'epoch': 30, 'FID': 0.028011006813232875, 'IS': 2.6021344305308145, 'samples': 5000}


In [ ]:
from pytorch_fid.inception import InceptionV3

class WrapperInceptionV3(nn.Module):
    def __init__(self, m):
        super().__init__(); self.m = m
    @torch.no_grad()
    def forward(self, x):
        return self.m(x)[0][:, :, 0, 0]

ckpt = torch.load(OUTPUT_DIR / "latest.pt", map_location=device, weights_only=False)
G.load_state_dict(ckpt["G"])

inc = InceptionV3([InceptionV3.BLOCK_INDEX_BY_DIM[2048]]).to(device).eval()
fid_metric = FID(num_features=2048, feature_extractor=WrapperInceptionV3(inc), device=device)
fid2048, is_score = evaluate_gan()
print(f"FID-2048: {fid2048:.2f} | IS: {is_score:.3f}")

Downloading: "https://github.com/mseitzer/pytorch-fid/releases/download/fid_weights/pt_inception-2015-12-05-6726825d.pth" to C:\Users\jia65/.cache\torch\hub\checkpoints\pt_inception-2015-12-05-6726825d.pth


100%|██████████| 91.2M/91.2M [00:01<00:00, 57.8MB/s]


Computing FID covariance distance; this can take a while...
FID-2048: 69.76 | IS: 2.602
